# 🐷 SwinV2 – Inference & Submission

Generates submission CSV from a trained SwinV2 checkpoint.  
Supports **TTA** (4 augmented views) and **multi-checkpoint ensembling**.

## ⚙️ Configuration

In [1]:
TAG = "T1"

DATA_ROOT   = "/datasets/multi-view-pig-posture-recognition"
TEST_CSV    = f"{DATA_ROOT}/test.csv"
IMG_DIR     = f"{DATA_ROOT}/test_images"

CKPT_PATHS  = [f"runs/swinv2_{TAG.lower()}/best_model.pth"]
OUTPUT_FILE = f"{TAG}_swinv2_submission.csv"

IMG_SIZE     = 256
BATCH_SIZE   = 32
NUM_WORKERS  = 8
USE_TTA      = True
PAD_RATIO    = 0.25

NUM_CLASSES  = 5
CLASS_NAMES  = ["Lateral_lying_left", "Lateral_lying_right",
                "Sitting", "Standing", "Sternal_lying"]

print(f"Tag: {TAG}  |  Output: {OUTPUT_FILE}  |  TTA: {USE_TTA}")
print(f"Checkpoints: {CKPT_PATHS}")

Tag: T1  |  Output: T1_swinv2_submission.csv  |  TTA: True
Checkpoints: ['runs/swinv2_t1/best_model.pth']


## 📚 Imports

In [2]:
import os, ast
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import torchvision.transforms as T
import timm

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

<jemalloc>: Unsupported system page size


Device: cuda


## 🗂️ Test Dataset & TTA Transforms

In [3]:
class PigTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.25):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform: crop = self.transform(crop)
        return crop, row["row_id"]

S = IMG_SIZE
NORM = [[0.485,0.456,0.406],[0.229,0.224,0.225]]

TTA_TRANSFORMS = [
    T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S, S)), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S+32, S+32)), T.CenterCrop(S), T.ToTensor(), T.Normalize(*NORM)]),
    T.Compose([T.Resize((S+32, S+32)), T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(*NORM)]),
]
print("Ready.")

Ready.


## 🔮 Run Inference

In [4]:
test_df = pd.read_csv(TEST_CSV)
print(f"Test instances: {len(test_df)}")

@torch.no_grad()
def predict_tta(model, df, img_dir, transforms):
    all_probs = []
    for i, tf in enumerate(transforms):
        ds     = PigTestDataset(df, img_dir, transform=tf, pad_ratio=PAD_RATIO)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        probs = []
        for imgs, _ in tqdm(loader, desc=f"  TTA {i+1}/{len(transforms)}", leave=False):
            with autocast():
                logits = model(imgs.to(DEVICE))
            probs.append(F.softmax(logits, dim=1).cpu().numpy())
        all_probs.append(np.vstack(probs))
    return np.mean(all_probs, axis=0)

transforms = TTA_TRANSFORMS if USE_TTA else [TTA_TRANSFORMS[0]]
ensemble_probs = []

for path in CKPT_PATHS:
    print(f"\nLoading: {path}")
    ckpt = torch.load(path, map_location="cpu")
    name = ckpt.get("model_name", "swinv2_base_window12to16_192to256.ms_in22k_ft_in1k")
    print(f"  Model: {name}  |  Val F1: {ckpt.get('val_f1', 0):.4f}")
    model = timm.create_model(name, pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(ckpt["model"])
    model.to(DEVICE).eval()
    ensemble_probs.append(predict_tta(model, test_df, IMG_DIR, transforms))
    del model; torch.cuda.empty_cache()

predictions = np.mean(ensemble_probs, axis=0).argmax(axis=1)
print(f"\n✓ Done – {len(predictions)} predictions")

Test instances: 11708

Loading: runs/swinv2_t1/best_model.pth
  Model: swinv2_base_window12to16_192to256.ms_in22k_ft_in1k  |  Val F1: 0.1978


  TTA 1/4:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 2/4:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 3/4:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 4/4:   0%|          | 0/366 [00:00<?, ?it/s]


✓ Done – 11708 predictions


## 📄 Save Submission

In [5]:
submission = pd.DataFrame({"row_id": test_df["row_id"].values, "class_id": predictions.astype(int)})
submission.to_csv(OUTPUT_FILE, index=False)
print(f"✓ Saved → {OUTPUT_FILE}  ({len(submission)} rows)")

assert list(submission.columns) == ["row_id", "class_id"]
assert set(submission["class_id"].unique()).issubset(set(range(5)))
assert len(submission) == len(test_df)
print("✓ Sanity checks passed")

print("\nPredicted distribution:")
counts = submission["class_id"].value_counts().sort_index()
for c in range(NUM_CLASSES):
    cnt = counts.get(c, 0)
    print(f"  {c} - {CLASS_NAMES[c]:<22} {cnt:>5}  ({100*cnt/len(submission):.1f}%)")

submission.head(10)

✓ Saved → T1_swinv2_submission.csv  (11708 rows)
✓ Sanity checks passed

Predicted distribution:
  0 - Lateral_lying_left         0  (0.0%)
  1 - Lateral_lying_right       27  (0.2%)
  2 - Sitting                    0  (0.0%)
  3 - Standing               11590  (99.0%)
  4 - Sternal_lying             91  (0.8%)


,row_id,class_id
0,test_pen1_tur_cam1_20250920_174649_0000,3
1,test_pen1_tur_cam1_20250920_174649_0001,3
2,test_pen1_tur_cam1_20250920_174649_0002,3
3,test_pen1_tur_cam1_20250920_174649_0003,3
4,test_pen1_tur_cam1_20250920_174649_0004,3
5,test_pen1_tur_cam1_20250920_174649_0005,3
6,test_pen1_tur_cam1_20250920_174649_0006,3
7,test_pen1_tur_cam1_20250920_174649_0007,3
8,test_pen1_tur_cam1_20250920_174649_0008,3
9,test_pen1_tur_cam1_20250921_050022_0000,3
